# Stage 6.5 — performance and scale clean-room checkpoint

This notebook validates Stage 6.5 from an exact 40-character Git commit. It creates an
isolated virtual environment, runs the canonical repository release gate, executes the full
Stage 6.5 performance profile once, validates its semantic-equivalence and measurement
contract, builds candidate paper assets, and emits a hashed checkpoint archive. Timing values
are intentionally environment-specific and are not expected to be byte-identical across runs.

Before selecting **Runtime → Run all**, paste the exact commit SHA into `REF` or leave it
blank and paste it when prompted. Branch names, mutable tags, `main`, and abbreviated SHAs
are not accepted.


In [ ]:
from __future__ import annotations

import re

REPO_URL = "https://github.com/richietrap/sbom_to_audit.git"
REF = ""  # Exact 40-character Stage 6.5 commit SHA.

if not REF.strip():
    REF = input("Paste the exact 40-character Stage 6.5 Git commit SHA: ").strip()
if re.fullmatch(r"[0-9a-fA-F]{40}", REF) is None:
    raise ValueError(
        "REF must be an exact 40-character Git commit SHA; branches, mutable tags, "
        "main, master, and abbreviated SHAs are not accepted."
    )

print("Repository:", REPO_URL)
print("Exact commit requested:", REF.lower())

In [ ]:
import hashlib
import json
import os
import platform
import re
import shlex
import shutil
import subprocess
import sys
import time
from collections.abc import Sequence
from pathlib import Path

STAGE = "6.5"
PACKAGE_VERSION = "0.6.5"
CHECKPOINT_ID = "STAGE6-5-CHECKPOINT-001"
WORKDIR = Path("/content/sbom_to_audit_stage65")
VENV = Path("/content/sbom_to_audit_stage65_venv")
LOG_ROOT = Path("/content/stage65_checkpoint_logs")
RELEASE_REPORT = Path("/content/stage65_release_validation.json")
RESULT_FULL = Path("/content/stage65_results")
ASSET_FULL = Path("/content/stage65_assets")
CHECKPOINT_ROOT = Path("/content/stage65_checkpoint_evidence")
ZIP_PATH = Path("/content/stage65_colab_checkpoint_evidence.zip")

for path in (WORKDIR, VENV, LOG_ROOT, RESULT_FULL, ASSET_FULL, CHECKPOINT_ROOT):
    if path.exists():
        shutil.rmtree(path)
for path in (RELEASE_REPORT, ZIP_PATH):
    if path.exists():
        path.unlink()
LOG_ROOT.mkdir(parents=True)


def run_checked(
    name: str,
    command: Sequence[str],
    *,
    cwd: Path | None = None,
    env: dict[str, str] | None = None,
    attempts: int = 1,
    retry_delay_seconds: int = 5,
) -> subprocess.CompletedProcess[str]:
    """Run one command and preserve stdout, stderr, return code, and each attempt."""

    safe_name = re.sub(r"[^A-Za-z0-9_.-]+", "_", name).strip("_").lower()
    command_list = [str(part) for part in command]
    records: list[dict[str, object]] = []
    last: subprocess.CompletedProcess[str] | None = None
    for attempt in range(1, attempts + 1):
        completed = subprocess.run(
            command_list,
            cwd=cwd,
            env=env,
            text=True,
            capture_output=True,
            check=False,
        )
        last = completed
        log_path = LOG_ROOT / f"{safe_name}.attempt-{attempt:02d}.log"
        log_path.write_text(
            "COMMAND: "
            + shlex.join(command_list)
            + f"\nWORKING DIRECTORY: {cwd or Path.cwd()}"
            + f"\nATTEMPT: {attempt}/{attempts}"
            + f"\nRETURN CODE: {completed.returncode}\n"
            + "\n--- STDOUT ---\n"
            + completed.stdout
            + "\n--- STDERR ---\n"
            + completed.stderr,
            encoding="utf-8",
        )
        records.append(
            {"attempt": attempt, "returncode": completed.returncode, "log": log_path.name}
        )
        (LOG_ROOT / f"{safe_name}.summary.json").write_text(
            json.dumps(
                {
                    "name": name,
                    "command": command_list,
                    "working_directory": str(cwd or Path.cwd()),
                    "attempts": records,
                    "final_returncode": completed.returncode,
                },
                indent=2,
                sort_keys=True,
            )
            + "\n",
            encoding="utf-8",
        )
        if completed.returncode == 0:
            print(f"PASS: {name}")
            return completed
        if attempt < attempts:
            time.sleep(retry_delay_seconds)
    assert last is not None
    diagnostic = (last.stdout + "\n" + last.stderr).strip()
    print(diagnostic[-4000:])
    raise RuntimeError(f"{name} failed; logs are under {LOG_ROOT}")


def tree_hashes(root: Path) -> dict[str, str]:
    return {
        path.relative_to(root).as_posix(): hashlib.sha256(path.read_bytes()).hexdigest()
        for path in sorted(root.rglob("*"))
        if path.is_file()
    }


run_checked("clone repository", ["git", "clone", "--no-checkout", REPO_URL, str(WORKDIR)])
run_checked("checkout exact commit", ["git", "checkout", "--detach", REF], cwd=WORKDIR)
commit = run_checked("resolve checked out commit", ["git", "rev-parse", "HEAD"], cwd=WORKDIR)
COMMIT = commit.stdout.strip().lower()
if COMMIT != REF.lower():
    raise RuntimeError(f"Checked out {COMMIT}, requested {REF.lower()}")
branch = run_checked("verify detached checkout", ["git", "branch", "--show-current"], cwd=WORKDIR)
if branch.stdout.strip():
    raise RuntimeError(f"Checkpoint is attached to a mutable branch: {branch.stdout.strip()}")
status = run_checked("verify clean checkout", ["git", "status", "--porcelain"], cwd=WORKDIR)
if status.stdout.strip():
    raise RuntimeError(f"Fresh checkout is dirty:\n{status.stdout}")
os.chdir(WORKDIR)
print("Exact commit checked out:", COMMIT)
print("Kernel Python:", sys.version)


In [ ]:
stdlib_venv = subprocess.run(
    [sys.executable, "-m", "venv", str(VENV)],
    text=True,
    capture_output=True,
    check=False,
)
(LOG_ROOT / "stdlib_venv_creation.log").write_text(
    "RETURN CODE: "
    + str(stdlib_venv.returncode)
    + "\n\n--- STDOUT ---\n"
    + stdlib_venv.stdout
    + "\n--- STDERR ---\n"
    + stdlib_venv.stderr,
    encoding="utf-8",
)
VENV_PYTHON = VENV / "bin" / "python"
pip_probe = subprocess.CompletedProcess(args=[], returncode=1, stdout="", stderr="not attempted")
if stdlib_venv.returncode == 0 and VENV_PYTHON.is_file():
    pip_probe = subprocess.run(
        [str(VENV_PYTHON), "-m", "pip", "--version"],
        text=True,
        capture_output=True,
        check=False,
    )
if stdlib_venv.returncode != 0 or pip_probe.returncode != 0:
    if VENV.exists():
        shutil.rmtree(VENV)
    run_checked(
        "install virtualenv fallback",
        [sys.executable, "-m", "pip", "install", "--no-cache-dir", "virtualenv"],
        attempts=3,
    )
    run_checked(
        "create isolated environment with virtualenv",
        [sys.executable, "-m", "virtualenv", str(VENV)],
    )

VENV_PYTHON = VENV / "bin" / "python"
if not VENV_PYTHON.is_file():
    raise FileNotFoundError(f"Isolated Python was not created: {VENV_PYTHON}")
VENV_ENV = os.environ.copy()
VENV_ENV["PATH"] = f"{VENV / 'bin'}:{VENV_ENV.get('PATH', '')}"
VENV_ENV["VIRTUAL_ENV"] = str(VENV)
VENV_ENV["PYTHONHASHSEED"] = "0"
VENV_ENV["PYTHONNOUSERSITE"] = "1"
VENV_ENV["PIP_DISABLE_PIP_VERSION_CHECK"] = "1"
VENV_ENV["PIP_REQUIRE_VIRTUALENV"] = "true"

run_checked(
    "upgrade isolated packaging tools",
    [str(VENV_PYTHON), "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel"],
    env=VENV_ENV,
    attempts=3,
)
run_checked(
    "install package and development dependencies",
    [str(VENV_PYTHON), "-m", "pip", "install", "--no-cache-dir", "-e", ".[dev]"],
    cwd=WORKDIR,
    env=VENV_ENV,
    attempts=3,
)
run_checked(
    "isolated dependency integrity",
    [str(VENV_PYTHON), "-m", "pip", "check"],
    cwd=WORKDIR,
    env=VENV_ENV,
)
probe = run_checked(
    "verify virtual environment isolation",
    [
        str(VENV_PYTHON),
        "-c",
        "import json, pathlib, site, sys; "
        "print(json.dumps({'prefix': str(pathlib.Path(sys.prefix).resolve()), "
        "'base_prefix': str(pathlib.Path(sys.base_prefix).resolve()), "
        "'user_site_enabled': site.ENABLE_USER_SITE}, sort_keys=True))",
    ],
    cwd=WORKDIR,
    env=VENV_ENV,
)
ISOLATION = json.loads(probe.stdout)
if Path(ISOLATION["prefix"]) != VENV.resolve() or ISOLATION["user_site_enabled"] is not False:
    raise RuntimeError(f"Virtual-environment isolation check failed: {ISOLATION}")

run_checked(
    "canonical release check",
    [str(VENV_PYTHON), "scripts/release_check.py", "--report", str(RELEASE_REPORT)],
    cwd=WORKDIR,
    env=VENV_ENV,
)
release = json.loads(RELEASE_REPORT.read_text(encoding="utf-8"))
if release.get("status") != "PASS":
    raise RuntimeError(f"Canonical release check did not pass: {release.get('errors')}")
print("Canonical release checks:", len(release.get("checks", [])))
print("Deterministic hashes:", len(release.get("deterministic_hashes", {})))

In [ ]:
run_checked(
    "Stage 6.5 full performance run",
    [
        str(VENV_PYTHON),
        "scripts/run_stage6_5_performance_remediation.py",
        "--profile",
        "full",
        "--destination",
        str(RESULT_FULL),
    ],
    cwd=WORKDIR,
    env=VENV_ENV,
)
run_checked(
    "validate Stage 6.5 full performance run",
    [
        str(VENV_PYTHON),
        "scripts/validate_stage6_5_performance_remediation.py",
        "--results",
        str(RESULT_FULL),
    ],
    cwd=WORKDIR,
    env=VENV_ENV,
)
run_checked(
    "build Stage 6.5 paper assets",
    [
        str(VENV_PYTHON),
        "scripts/build_stage6_5_paper_assets.py",
        "--results",
        str(RESULT_FULL),
        "--destination",
        str(ASSET_FULL),
    ],
    cwd=WORKDIR,
    env=VENV_ENV,
)

stage65_report = json.loads(
    (RESULT_FULL / "stage6_5_performance_report.json").read_text(encoding="utf-8")
)
if stage65_report.get("evaluation_status") != "CANDIDATE_NOT_FROZEN":
    raise RuntimeError("Stage 6.5 status boundary changed unexpectedly")
if stage65_report.get("manuscript_eligible") is not False:
    raise RuntimeError("Stage 6.5 candidate must not be manuscript-eligible")
if stage65_report.get("all_decision_equivalent") is not True:
    raise RuntimeError("Stage 6.5 scale workload changed accepted decision semantics")
if stage65_report.get("source_commit") != COMMIT:
    raise RuntimeError(
        f"Stage 6.5 report recorded {stage65_report.get('source_commit')}, expected {COMMIT}"
    )
if stage65_report.get("source_tree_dirty") is not False:
    raise RuntimeError("Exact-commit Stage 6.5 performance run did not record a clean tree")
print("Stage 6.5 workload count:", stage65_report["workload_count"])
print("Stage 6.5 measured runs per workload:", stage65_report["measured_runs"])
print("Stage 6.5 environment:", stage65_report["environment"])


In [ ]:
import zipfile
from datetime import datetime, timezone

CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=False)
shutil.copy2(RELEASE_REPORT, CHECKPOINT_ROOT / RELEASE_REPORT.name)
shutil.copytree(LOG_ROOT, CHECKPOINT_ROOT / "command_logs")
shutil.copytree(RESULT_FULL, CHECKPOINT_ROOT / "stage6_5_results")
shutil.copytree(ASSET_FULL, CHECKPOINT_ROOT / "stage6_5_paper_assets")
for relative in (
    "evaluation/stage6_4_performance_protocol_v0.2.yaml",
    "evaluation/stage6_5_remediation_protocol_v0.1.yaml",
    "evaluation/stage6_3_mutation_protocol_v0.1.yaml",
    "evaluation/stage6_2_robustness_protocol_v0.1.yaml",
    "evaluation/freeze/stage6_1_protocol_freeze.json",
    "evaluation/baseline_protocol_v0.2.yaml",
):
    source = WORKDIR / relative
    target = CHECKPOINT_ROOT / "registered_controls" / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, target)

isolated_python = run_checked(
    "record isolated Python version",
    [str(VENV_PYTHON), "--version"],
    cwd=WORKDIR,
    env=VENV_ENV,
).stdout.strip()
environment = {
    "checkpoint_id": CHECKPOINT_ID,
    "checkpoint_status": "PASS",
    "stage": STAGE,
    "package_version": PACKAGE_VERSION,
    "repository": REPO_URL,
    "requested_ref": REF.lower(),
    "git_commit": COMMIT,
    "generated_at": datetime.now(timezone.utc).isoformat().replace("+00:00", "Z"),
    "kernel_python": sys.version,
    "isolated_python": isolated_python,
    "isolation": ISOLATION,
    "platform": platform.platform(),
    "release_status": release["status"],
    "release_check_count": len(release.get("checks", [])),
    "deterministic_hash_count": len(release.get("deterministic_hashes", {})),
    "stage6_5_status": stage65_report["evaluation_status"],
    "stage6_5_manuscript_eligible": stage65_report["manuscript_eligible"],
    "stage6_5_workload_count": stage65_report["workload_count"],
    "stage6_5_measurement_environment": stage65_report["environment"],
    "timing_non_determinism_boundary": (
        "Observed latency and memory values are environment-specific and are not required "
        "to match values from other machines or runs."
    ),
}
(CHECKPOINT_ROOT / "checkpoint_environment.json").write_text(
    json.dumps(environment, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)

checksums = tree_hashes(CHECKPOINT_ROOT)
(CHECKPOINT_ROOT / "evidence_checksums.json").write_text(
    json.dumps(
        {
            "algorithm": "sha256",
            "scope": "all checkpoint evidence files except this inventory and the outer ZIP",
            "files": checksums,
        },
        indent=2,
        sort_keys=True,
    )
    + "\n",
    encoding="utf-8",
)
for relative, digest in checksums.items():
    if hashlib.sha256((CHECKPOINT_ROOT / relative).read_bytes()).hexdigest() != digest:
        raise RuntimeError(f"Evidence changed during checksum generation: {relative}")

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(CHECKPOINT_ROOT.rglob("*")):
        if path.is_file():
            archive.write(path, path.relative_to(CHECKPOINT_ROOT))
with zipfile.ZipFile(ZIP_PATH) as archive:
    corrupt = archive.testzip()
    if corrupt is not None:
        raise RuntimeError(f"Checkpoint ZIP integrity failure: {corrupt}")
    names = archive.namelist()
    if len(names) != len(set(names)):
        raise RuntimeError("Checkpoint ZIP contains duplicate member paths")
    required = {
        "checkpoint_environment.json",
        "evidence_checksums.json",
        RELEASE_REPORT.name,
        "stage6_5_results/stage6_5_performance_report.json",
        "stage6_5_results/stage6_5_scale_summary.csv",
        "stage6_5_results/stage6_5_raw_trials.csv",
        "stage6_5_results/stage6_5_memory_observations.csv",
        "stage6_5_paper_assets/data/stage6_5_asset_manifest.json",
        "registered_controls/evaluation/stage6_4_performance_protocol_v0.2.yaml",
    }
    missing = sorted(required - set(names))
    if missing:
        raise RuntimeError(f"Checkpoint ZIP is missing required evidence: {missing}")

zip_hash = hashlib.sha256(ZIP_PATH.read_bytes()).hexdigest()
print("Checkpoint status: PASS")
print("Exact Git commit:", COMMIT)
print("Evidence ZIP:", ZIP_PATH)
print("Evidence ZIP SHA-256:", zip_hash)


## Acceptance boundary

A successful run must finish with `Checkpoint status: PASS`, print the exact Git commit, and
produce `/content/stage65_colab_checkpoint_evidence.zip` with its SHA-256. Preserve that ZIP
unchanged. Stage 6.5 performance observations remain `CANDIDATE_NOT_FROZEN` and are not final
manuscript evidence until the later evaluation freeze.
